# Banking agent: VAE + GAN + LoRA (PEFT) on Google Colab

Runs the three new modules. Use a **GPU runtime** (Runtime -> Change runtime type -> T4 GPU).

1. Setup (clone the repository, install packages)
2. VAE + GAN experiment (`src.augment`) - works on CPU or GPU
3. LoRA fine-tuning of Qwen2.5-1.5B-Instruct (`src.finetune`) - needs the GPU
4. Compare RAG vs LoRA on the development and held-out subsets
5. Optional: RAG with the synthetic pool, and the out-of-scope guard

## 1. Setup
Clones the repository from GitHub into the Colab session and installs the packages. Push your latest changes first: the clone takes what is on GitHub. (Fallback: if a `banking-agent.zip` is uploaded to `/content`, it is used instead.)

In [ ]:
import os, zipfile
if os.path.exists('/content/banking-agent.zip'):
    zipfile.ZipFile('/content/banking-agent.zip').extractall('/content')
elif not os.path.exists('/content/banking-agent'):
    !git clone https://github.com/2025paml556-bhoomika/banking-agent.git /content/banking-agent
%cd /content/banking-agent
!pip -q install transformers peft accelerate jsonschema scikit-learn scipy pandas gradio
# optional, only for QLoRA (4-bit): !pip -q install bitsandbytes

In [ ]:
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. VAE + GAN
Trains the conditional text VAE and the latent GAN, generates synthetic messages, filters them, and measures the effect on a classifier (full data and 10 % low-resource). Saves `models/vae.pt`, `models/gan.pt`, `data/synthetic_*.csv`, `evaluation/results/vae_gan_results.json`.

In [ ]:
!python -m src.augment --vae-epochs 30 --gan-steps 3000 --per-intent 30 --low-seeds 3

In [ ]:
import json, pandas as pd
R = json.load(open('evaluation/results/vae_gan_results.json'))
print(json.dumps({k: R[k] for k in ['gen_stats','latent_frechet','kept_after_filter','downstream_full','knn5_vote']}, indent=2))
print(pd.read_csv('evaluation/results/samples_vae.csv').head(10).to_string())
print(pd.read_csv('evaluation/results/samples_gan.csv').head(10).to_string())

## 3. LoRA fine-tuning (PEFT)
`src.finetune` trains a rank-8 adapter on the retrieval prompt. It holds out 500 of the 10,003 train rows for validation, runs up to 3 epochs with early stopping, and saves the adapter to `models/intent-lora-retrieval/`. The base model stays frozen. This took 2 h 53 min on an RTX 3060.

**Runtime note:** training uses bfloat16, which the free T4 GPU does not support. Use an **L4 or A100 runtime** for this cell. Everything else in the notebook runs on a T4 or CPU.

In [ ]:
!python -m src.finetune

In [ ]:
import os; print(sorted(os.listdir('models/intent-lora-retrieval')))  # adapter_config.json and adapter_model.safetensors confirm the save

## 4. Compare the classifiers
`evaluation.final_eval` scores the RAG classifier on the 100 held-out rows. `evaluation.finetune_eval` scores the LoRA adapter on the development subset and the held-out subset, and writes `finetune_retrieval_scores.md`. Each run is a separate process because `src/llm.py` loads the model at import time.

In [ ]:
!python -m evaluation.final_eval
!python -m evaluation.finetune_eval

In [ ]:
import pandas as pd
for f in ['results_heldout','results_finetune_retrieval_dev','results_finetune_retrieval_heldout']:
    d = pd.read_csv(f'evaluation/results/{f}.csv'); print(f, 'accuracy', d['correct'].mean(), 'no valid intent', int((d['predicted'].isna() | (d['predicted']=='')).sum()))

## 5. Optional: RAG with the synthetic pool, and the VAE guard

In [ ]:
!RETRIEVAL_POOL=vae+gan python -m src.retrieval

In [ ]:
!python - <<'EOF'
from src.guard import vae_score
for m in ['I lost my card', 'What is the weather in Paris?', 'How long does a transfer take?', 'Write me a poem']:
    print(m, vae_score(m))
EOF